# Controllable Costs Upload

Cleans the existing controllable-cost upload workflow and handles the 2026 files correctly:

- 2025 historical files are loaded from sheets `1` through `7`.
- 2026 Q1 uses March YTD actuals.
- 2026 Q2 uses June YTD actuals **minus Q1 March YTD actuals**, so Q2 contains only incremental April–June cost.
- The final dataset is aligned to one schema and can be uploaded to `qmi.controllable_costs`.

In [ ]:
import os
import urllib.parse

import pandas as pd
import pyodbc
from sqlalchemy import create_engine

## Database connection

This avoids keeping a password directly in the notebook. Set the environment variable
`CONTROLLABLE_COSTS_ODBC_CONNECTION` to the same ODBC connection string you were using before.

Example format:

`DRIVER={ODBC Driver 18 for SQL Server};SERVER=...;DATABASE=...;UID=...;PWD=...;TrustServerCertificate=yes`

In [ ]:
odbc_connection = os.getenv("CONTROLLABLE_COSTS_ODBC_CONNECTION")

engine = None
if odbc_connection:
    params = urllib.parse.quote_plus(odbc_connection)
    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
else:
    print(
        "Database connection not configured. "
        "Data-prep cells will still run; set CONTROLLABLE_COSTS_ODBC_CONNECTION before uploading."
    )

## Helpers

In [ ]:
OUTPUT_COLUMNS = [
    "Cost Category",
    "Address",
    "Cost Element",
    "Cost Element Description",
    "Cost",
    "Quarter",
    "Year",
]

MATCH_COLUMNS = [
    "Cost Category",
    "Address",
    "Cost Element",
    "Cost Element Description",
]


def clean_cost(series):
    """Convert currency-like values to numeric without failing on commas or blanks."""
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce",
    )


def normalize_cost_element(series):
    """Keep numeric cost elements as nullable integers while preserving missing values."""
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.round().astype("Int64")


def load_2026_ytd(path, sheet_name, ytd_column, quarter):
    raw = pd.read_excel(path, sheet_name=sheet_name)
    df = raw.copy()

    # Some report-level rows do not have a G/L Account2 description.
    df["G/L Account2"] = df["G/L Account2"].fillna(df["Report Level1"])

    df = (
        df[
            [
                "Address",
                "CORP FAC Category",
                "G/L Account",
                "G/L Account2",
                ytd_column,
            ]
        ]
        .rename(
            columns={
                "CORP FAC Category": "Cost Category",
                "G/L Account": "Cost Element",
                "G/L Account2": "Cost Element Description",
                ytd_column: "Cost",
            }
        )
        .copy()
    )

    df["Cost"] = clean_cost(df["Cost"])
    df = df.dropna(subset=["Cost"]).copy()

    df["Cost Element"] = normalize_cost_element(df["Cost Element"])
    df["Quarter"] = quarter
    df["Year"] = 2026

    return df[OUTPUT_COLUMNS].reset_index(drop=True)

## 2025 historical data

In [ ]:
historical_path = "data/controllable_costs.xlsx"

frames = []
for sheet in map(str, range(1, 8)):
    df = pd.read_excel(
        historical_path,
        sheet_name=sheet,
        skiprows=5,
        dtype={"Cost": "string", "Cost Element": "string"},
    )

    df = df[
        [
            "CRE Cost Category",
            "Address",
            "Cost Element",
            "Cost Element Description",
            "Cost",
            "Period",
        ]
    ].copy()

    frames.append(df)

costs = pd.concat(frames, ignore_index=True)

costs["Cost"] = clean_cost(costs["Cost"])
costs = costs.dropna(subset=["Cost"]).copy()

costs["Quarter"] = costs["Period"].astype("string").str[:2]
costs["Year"] = pd.to_numeric(
    costs["Period"].astype("string").str[3:],
    errors="coerce",
).astype("Int64")

costs = costs.drop(columns="Period").rename(
    columns={"CRE Cost Category": "Cost Category"}
)

costs["Cost Element"] = normalize_cost_element(costs["Cost Element"])
costs = costs[OUTPUT_COLUMNS].reset_index(drop=True)

costs.head(3)

## 2026 Q1 — March YTD

In [ ]:
q1 = load_2026_ytd(
    path="data/controllable_costs_q1_2026.xlsx",
    sheet_name="DS Site Detail",
    ytd_column="YTD Mar Actual",
    quarter="Q1",
)

q1.head(3)

## 2026 Q2 — June YTD converted to quarter-only

In [ ]:
q2_ytd = load_2026_ytd(
    path="data/controllable_costs_q2_2026.xlsx",
    sheet_name="DS Site YTD Actuals",
    ytd_column="YTD Jun Actual",
    quarter="Q2",
)

# Aggregate Q1 first in case the source contains duplicate rows at the same reporting grain.
q1_for_subtraction = (
    q1.groupby(MATCH_COLUMNS, dropna=False, as_index=False)["Cost"]
    .sum()
    .rename(columns={"Cost": "Q1 YTD Cost"})
)

q2 = q2_ytd.merge(
    q1_for_subtraction,
    on=MATCH_COLUMNS,
    how="left",
    validate="many_to_one",
)

q2["Q1 YTD Cost"] = q2["Q1 YTD Cost"].fillna(0)
q2["Q2 YTD Cost"] = q2["Cost"]

# Q2 source is YTD through June, so remove Q1 YTD through March.
q2["Cost"] = (q2["Q2 YTD Cost"] - q2["Q1 YTD Cost"]).round(2)

q2 = q2[OUTPUT_COLUMNS].reset_index(drop=True)

q2.head(3)

## Sanity checks

In [ ]:
print(f"2025 rows: {len(costs):,}")
print(f"2026 Q1 rows: {len(q1):,}")
print(f"2026 Q2 rows: {len(q2):,}")

print(f"2026 Q1 total: {q1['Cost'].sum():,.2f}")
print(f"2026 Q2 quarter-only total: {q2['Cost'].sum():,.2f}")
print(f"2026 H1 total (Q1 + Q2): {(q1['Cost'].sum() + q2['Cost'].sum()):,.2f}")
print(f"Q2 June-YTD source total: {q2_ytd['Cost'].sum():,.2f}")

difference = (
    q1["Cost"].sum()
    + q2["Cost"].sum()
    - q2_ytd["Cost"].sum()
)
print(f"Reconciliation difference: {difference:,.2f}")

## Combine and upload

In [ ]:
concatted = pd.concat([costs, q1, q2], ignore_index=True)

concatted["Cost Category"] = concatted["Cost Category"].fillna("0 Other")
concatted["Year"] = concatted["Year"].astype(int)
concatted["Cost"] = concatted["Cost"].astype(float).round(2)

concatted.head()

In [ ]:
# Optional local check/export before replacing the SQL table.
concatted.to_excel(
    "data/controllable_costs_processed.xlsx",
    index=False,
)

In [ ]:
if engine is None:
    raise RuntimeError(
        "No database engine is configured. "
        "Set CONTROLLABLE_COSTS_ODBC_CONNECTION and rerun the connection cell."
    )

concatted.to_sql(
    "controllable_costs",
    schema="qmi",
    con=engine,
    if_exists="replace",
    index=False,
)

print(f"Uploaded {len(concatted):,} rows to qmi.controllable_costs")